# Hyperparameter sweep on HAM10000 — Weights & Biases

A **W&B Sweep** is W&B's own hyperparameter optimiser (the alternative to Optuna).
You declare a search space; W&B's Bayesian optimiser proposes configs, this
notebook trains one model per config, and the **website** draws the
parallel-coordinates and parameter-importance plots that tell you *which
hyperparameters actually matter*.

**How to run on Kaggle**
1. Add the dataset: right sidebar -> *Add Input* -> **Skin Cancer MNIST: HAM10000** (by kmader).
2. Add your W&B key: *Add-ons -> Secrets -> `WANDB_API_KEY`* (from https://wandb.ai/authorize), tick this notebook.
3. *Settings -> Accelerator -> GPU T4 x2*, then **Run All**.
4. Watch it live at https://wandb.ai/ -> project `skin-lesion` -> **Sweeps** (left sidebar).

To keep each trial short enough for Kaggle's GPU quota, the sweep trains a
**lighter, faster proxy** (EfficientNet-B3 @ 224px, capped data, few epochs) — the
goal is to *rank* hyperparameters cheaply, then retrain the winner at full
resolution in `kaggle_training.ipynb`.

## 1. Environment

In [ ]:
# Kaggle ships torch, torchvision, scikit-learn, matplotlib, pillow — nothing to install.
import os
import json
import time
import csv
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

IN_KAGGLE = Path("/kaggle/input").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
USE_AMP = device.type == "cuda"  # mixed precision: big speed/memory win on GPU, no-op elsewhere

# Mixed-precision helpers, tolerant of the torch.amp (>=2.3) vs torch.cuda.amp API split.
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler():
        return _GradScaler(device.type, enabled=USE_AMP)
    def amp_ctx():
        return _autocast(device.type, enabled=USE_AMP)
except Exception:  # older torch
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler():
        return _GradScaler(enabled=USE_AMP)
    def amp_ctx():
        return _autocast(enabled=USE_AMP)

print(f"PyTorch {torch.__version__}")
print(f"Kaggle: {IN_KAGGLE}  device: {device}  AMP: {USE_AMP}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Fail fast on an unsupported GPU. Kaggle's P100 is sm_60, but recent PyTorch
    # CUDA builds only ship kernels for sm_70+ — without this check the run would
    # crash with a cryptic error deep inside the first forward pass instead.
    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    supported = torch.cuda.get_arch_list()
    if supported and arch not in supported:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(0)} ({arch}) is not supported by this "
            f"PyTorch build (supports {supported}). On Kaggle, switch "
            "Settings -> Accelerator -> GPU T4 x2 (the T4 is sm_75) and Run All again."
        )
elif IN_KAGGLE:
    print("WARNING: no GPU detected. Settings -> Accelerator -> GPU T4 x2, then Run All again.")

## 2. Experiment tracking (Weights & Biases)

In [ ]:
# Experiment-tracking switches (defined here so this cell is self-contained
# and runs before anything else needs them). Set USE_WANDB=False to run the
# pipeline with no tracking.
USE_WANDB = True
WANDB_PROJECT = "skin-lesion"

# --- Weights & Biases setup (safe no-op when USE_WANDB = False) --------------
# Auth on Kaggle WITHOUT putting your key in the notebook:
#   Add-ons -> Secrets -> add WANDB_API_KEY (from https://wandb.ai/authorize),
#   then tick this notebook to grant access. Running locally, an env var
#   WANDB_API_KEY (or a prior `wandb login`) is picked up automatically.
wandb = None
if USE_WANDB:
    try:
        import wandb
    except ImportError:
        import subprocess, sys as _sys
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
        import wandb

    _key = os.environ.get("WANDB_API_KEY")
    if _key is None and IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            _key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception as e:
            print(f"No WANDB_API_KEY secret available ({e}).")
    if _key:
        _key = _key.strip()  # guard against a stray newline/space in the secret
    if _key and len(_key) >= 40:
        wandb.login(key=_key)
        print(f"wandb: logged in ({len(_key)}-char key), live tracking ON.")
    elif _key:
        raise ValueError(
            f"WANDB_API_KEY looks invalid: {len(_key)} characters (a real key is "
            "40+). Re-copy the FULL key from https://wandb.ai/authorize into the "
            "Kaggle secret WANDB_API_KEY — a partial paste is the usual cause.")
    else:
        print("wandb: no API key found -> tracking OFF (training still runs).")
        USE_WANDB = False


## 3. Fixed configuration (everything the sweep does NOT vary)

In [ ]:
# --- paths ------------------------------------------------------------------
if IN_KAGGLE:
    INPUT_DIR = Path("/kaggle/input/skin-cancer-mnist-ham10000")
    OUT_DIR = Path("/kaggle/working/results")
else:
    INPUT_DIR = Path.cwd() / "data" / "ham10000_raw"
    OUT_DIR = Path.cwd() / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_TO_CLASS = {
    "akiec": "actinic_keratoses", "bcc": "basal_cell_carcinoma",
    "bkl": "benign_keratosis-like_lesions", "df": "dermatofibroma",
    "nv": "melanocytic_nevi", "mel": "melanoma", "vasc": "vascular_lesions",
}

# --- fixed (NOT swept) ------------------------------------------------------
# A deliberately cheap proxy so a whole sweep fits in Kaggle's GPU budget. The
# sweep ranks hyperparameters; you then retrain the winner at full res elsewhere.
MODEL_NAME = "efficientnet_b3"     # fast, strong backbone; fixed for the sweep
IMG_SIZE = 224                     # 224 trains ~2x faster than 300/384
VAL_RESIZE = round(IMG_SIZE * 256 / 224)
NUM_WORKERS = min(4, os.cpu_count() or 2)

HEAD_EPOCHS = 2                    # short head warm-up
FT_EPOCHS = 12                     # max fine-tune epochs (early stopping cuts it)
EARLY_STOP_PATIENCE = 3

# Data caps that make a trial take minutes, not an hour.
SEED = 42
VAL_FRACTION = 0.15
VAL_CAP = 100                      # val images per class
TRAIN_CAP = 500                    # train images per class (~3.5k total)

N_TRIALS = 15                      # how many configs the agent will try

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Sweep proxy: {MODEL_NAME} @ {IMG_SIZE}px, {N_TRIALS} trials, "
      f"caps train<={TRAIN_CAP}/val<={VAL_CAP} per class")

## 4. Build the train / val split (runs once, shared by every trial)

In [ ]:
# 0) locate the dataset. The mount slug / sub-folder layout can vary, so find the
#    HAM10000_metadata.csv anywhere under /kaggle/input and use its folder as root.
search_roots = [INPUT_DIR]
if IN_KAGGLE:
    search_roots.append(Path("/kaggle/input"))
meta_csv = None
for root in search_roots:
    if root.exists():
        meta_csv = next(root.rglob("HAM10000_metadata*.csv"), None)
        if meta_csv is not None:
            break
if meta_csv is None:
    listing = "\n".join(f"  {p}" for p in sorted(Path('/kaggle/input').glob('*'))) \
        if Path('/kaggle/input').exists() else "  (/kaggle/input does not exist)"
    raise FileNotFoundError(
        "Could not find HAM10000_metadata.csv. Add the dataset via the right "
        "sidebar -> Add Input -> 'Skin Cancer MNIST: HAM10000' (by kmader).\n"
        f"Currently mounted under /kaggle/input:\n{listing}"
    )
INPUT_DIR = meta_csv.parent
print(f"Dataset root: {INPUT_DIR}")

# 1) index every image file by its image_id (extensions/case vary across mirrors)
img_paths = {}
for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
    for p in INPUT_DIR.rglob(ext):
        img_paths.setdefault(p.stem, p)
print(f"Indexed {len(img_paths)} image files under {INPUT_DIR}")
assert img_paths, f"Found metadata but no image files under {INPUT_DIR} — check the dataset is complete."

# 2) read the metadata (image_id, lesion_id, dx, ...)
with open(meta_csv, newline="") as f:
    rows = list(csv.DictReader(f))
print(f"Metadata: {len(rows)} rows ({meta_csv.name})")

# 3) group images by lesion so the same lesion can't appear in both splits
lesion_imgs = defaultdict(list)   # lesion_id -> [image_id, ...]
lesion_dx = {}                    # lesion_id -> dx code
for r in rows:
    iid, lid, dx = r["image_id"], r["lesion_id"], r["dx"].lower()
    if iid not in img_paths:
        continue  # metadata row with no matching image file
    lesion_imgs[lid].append(iid)
    lesion_dx[lid] = dx

classes = sorted(DX_TO_CLASS[dx] for dx in set(lesion_dx.values()))
class_to_idx = {c: i for i, c in enumerate(classes)}
assert len(classes) == 7, f"Expected 7 classes, got {classes}"

lesions_by_class = defaultdict(list)
for lid, dx in lesion_dx.items():
    lesions_by_class[DX_TO_CLASS[dx]].append(lid)

# 4) per-class grouped split
rng = random.Random(SEED)
train_samples, val_samples = [], []   # each: (Path, class_idx)
for cls in classes:
    lesions = lesions_by_class[cls]
    rng.shuffle(lesions)
    total_imgs = sum(len(lesion_imgs[l]) for l in lesions)
    val_target = min(VAL_CAP, max(1, round(VAL_FRACTION * total_imgs)))
    val_n, train_n = 0, 0
    for lid in lesions:
        imgs = [(img_paths[iid], class_to_idx[cls]) for iid in lesion_imgs[lid]]
        if val_n < val_target:
            val_samples.extend(imgs)
            val_n += len(imgs)
        elif train_n < TRAIN_CAP:
            take = imgs[: max(0, TRAIN_CAP - train_n)]
            train_samples.extend(take)
            train_n += len(take)

rng.shuffle(train_samples)
print(f"Classes: {classes}")
print(f"Train: {len(train_samples)} images   Val: {len(val_samples)} images")
for cls in classes:
    ci = class_to_idx[cls]
    tr = sum(1 for _, y in train_samples if y == ci)
    va = sum(1 for _, y in val_samples if y == ci)
    print(f"  {cls:30s} train {tr:5d}  val {va:4d}")

## 5. Datasets, model helpers, evaluation (defined once)

In [ ]:
class SkinDataset(Dataset):
    """ImageFolder-compatible dataset over an explicit (path, label) list.

    Exposes `.samples` and `.classes` so the class-weight helper and the rest of
    the notebook work unchanged. Reads images straight from the read-only Kaggle
    input — the HAM10000 jpgs are already 600x450, large enough for 384px inputs,
    so there is no need to copy or pre-resize ~2.6GB of files."""
    def __init__(self, samples, transform, classes):
        self.samples = samples
        self.transform = transform
        self.classes = classes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),  # lesions have no canonical orientation
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize(VAL_RESIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def class_weights(samples, n_classes):
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    weights = counts.sum() / (n_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)


def log_class_prior(samples, n_classes):
    """log P(y) over the training set — the offset logit adjustment subtracts."""
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    prior = counts / counts.sum()
    return torch.log(torch.tensor(prior, dtype=torch.float32).clamp_min(1e-12))


class LogitAdjustedLoss(nn.Module):
    """Logit-adjusted cross-entropy (Menon et al., ICLR 2021).

    Trains on ``logits + tau * log_prior``, which is equivalent to enforcing a
    per-class margin proportional to the label frequency. Because the prior is
    added during training, the *raw* logits at inference are already corrected
    for the class imbalance — so the app and analysis need no change. This is an
    alternative to inverse-frequency class weighting, not an addition to it."""
    def __init__(self, log_prior, tau):
        super().__init__()
        self.register_buffer("adj", tau * log_prior)

    def forward(self, logits, target):
        return F.cross_entropy(logits + self.adj, target)


def balanced_sampler(samples, n_classes):
    """A WeightedRandomSampler that draws each class with equal probability, for
    the cRT phase. Sampling (rather than capping) keeps every image available
    while equalising how often each class is seen per epoch."""
    counts = np.bincount([y for _, y in samples], minlength=n_classes)
    per_class_w = 1.0 / np.maximum(counts, 1)
    sample_w = [per_class_w[y] for _, y in samples]
    return WeightedRandomSampler(sample_w, num_samples=len(samples), replacement=True)


def replace_head(model, n_classes):
    """Swap the final classification layer for a fresh n_classes one.

    Different torchvision families name the head differently, so we locate the
    last nn.Linear generically. Works for EfficientNet/ConvNeXt (.classifier),
    Swin (.head), ViT (.heads), ResNet (.fc). Mirrors skin/skin_model.replace_head."""
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is None:
            continue
        if isinstance(head, nn.Linear):
            setattr(model, head_attr, nn.Linear(head.in_features, n_classes))
            return model
        last_linear_name = None
        for name, m in head.named_modules():
            if isinstance(m, nn.Linear):
                last_linear_name = name
        if last_linear_name is not None:
            parent = head
            *path, leaf = last_linear_name.split(".")
            for p in path:
                parent = getattr(parent, p)
            in_features = getattr(parent, leaf).in_features
            setattr(parent, leaf, nn.Linear(in_features, n_classes))
            return model
    raise ValueError(f"Could not find a classifier head on {type(model).__name__}")


def freeze_backbone(model, freeze=True):
    """Phase-1 helper: freeze everything except the classification head."""
    head_params = set()
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is not None:
            head_params.update(id(p) for p in head.parameters())
    for p in model.parameters():
        p.requires_grad = (id(p) in head_params) if freeze else True


def head_parameters(model):
    for head_attr in ["classifier", "head", "heads", "fc"]:
        head = getattr(model, head_attr, None)
        if head is not None:
            return head.parameters()
    return model.parameters()


# Datasets built once; per-trial DataLoaders are rebuilt inside train_run because
# batch_size is swept.
train_ds = SkinDataset(train_samples, train_tf, classes)
val_ds = SkinDataset(val_samples, val_tf, classes)
CKPT_PATH = OUT_DIR / f"{MODEL_NAME}_best.pt"
print(f"Datasets ready: {len(train_ds)} train / {len(val_ds)} val")


In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad(), amp_ctx():
        for x, y in loader:
            logits = model(x.to(device, non_blocking=True))
            preds.append(logits.argmax(1).cpu().numpy())
            targets.append(y.numpy())
    preds, targets = np.concatenate(preds), np.concatenate(targets)
    return preds, targets, balanced_accuracy_score(targets, preds)

## 6. The training function the sweep agent calls

One call = one trial. It reads the proposed hyperparameters from `wandb.config`,
trains the three phases (head -> fine-tune -> optional cRT), and logs
`best_val_bal_acc` — the value the sweep maximises.

In [ ]:
def run_phase(model, loader, criterion, scaler, params_fn, lr, weight_decay,
              epochs, tag, state, patience=None):
    """Train one phase; log every epoch to wandb; track the running best in
    `state` (shared across phases) so early stopping and the sweep metric see a
    single monotonic `best_val_bal_acc`."""
    opt = torch.optim.AdamW(params_fn(), lr=lr, weight_decay=weight_decay)
    since_improve = 0
    for ep in range(epochs):
        model.train()
        running, n = 0.0, 0
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            opt.zero_grad()
            with amp_ctx():
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            running += loss.item() * len(y); n += len(y)
        _, _, bal = evaluate(model, val_loader_for_trial)
        state["step"] += 1
        improved = bal > state["best"]
        if improved:
            state["best"] = bal; since_improve = 0
        else:
            since_improve += 1
        wandb.log({"train_loss": running / n, "val_bal_acc": bal,
                   "best_val_bal_acc": state["best"], "phase": tag,
                   "phase_epoch": ep}, step=state["step"])
        if patience is not None and since_improve >= patience:
            break


def train_run():
    """A single sweep trial. wandb.init() with no args picks up the config the
    sweep agent assigned to this trial."""
    global val_loader_for_trial
    run = wandb.init()          # config provided by the sweep agent
    cfg = wandb.config

    pin = device.type == "cuda"
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=pin, persistent_workers=NUM_WORKERS > 0)
    val_loader_for_trial = DataLoader(
        val_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=pin, persistent_workers=NUM_WORKERS > 0)

    model = models.get_model(MODEL_NAME, weights="IMAGENET1K_V1")
    model = replace_head(model, len(classes)).to(device)

    if cfg.logit_adjust_tau > 0:
        criterion = LogitAdjustedLoss(
            log_class_prior(train_samples, len(classes)), cfg.logit_adjust_tau).to(device)
    else:
        criterion = nn.CrossEntropyLoss(
            weight=class_weights(train_samples, len(classes)).to(device))

    scaler = make_scaler()
    state = {"best": -1.0, "step": 0}

    freeze_backbone(model, freeze=True)
    run_phase(model, train_loader, criterion, scaler,
              lambda: head_parameters(model), cfg.head_lr, cfg.weight_decay,
              HEAD_EPOCHS, "head", state)
    freeze_backbone(model, freeze=False)
    run_phase(model, train_loader, criterion, scaler,
              lambda: model.parameters(), cfg.ft_lr, cfg.weight_decay,
              FT_EPOCHS, "ft", state, patience=EARLY_STOP_PATIENCE)
    if cfg.crt_epochs > 0:
        crt_loader = DataLoader(
            train_ds, batch_size=cfg.batch_size,
            sampler=balanced_sampler(train_samples, len(classes)),
            num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=NUM_WORKERS > 0)
        freeze_backbone(model, freeze=True)
        run_phase(model, crt_loader, nn.CrossEntropyLoss(), scaler,
                  lambda: head_parameters(model), cfg.crt_lr, cfg.weight_decay,
                  cfg.crt_epochs, "crt", state)

    run.summary["best_val_bal_acc"] = state["best"]
    wandb.finish()

## 7. Define the search space and launch the sweep

In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "best_val_bal_acc", "goal": "maximize"},
    "parameters": {
        "ft_lr":            {"distribution": "log_uniform_values", "min": 1e-5, "max": 3e-4},
        "head_lr":          {"distribution": "log_uniform_values", "min": 3e-4, "max": 5e-3},
        "weight_decay":     {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-3},
        "logit_adjust_tau": {"values": [0.0, 0.5, 1.0, 1.5]},
        "batch_size":       {"values": [16, 32]},
        "crt_epochs":       {"values": [0, 3]},
        "crt_lr":           {"value": 1e-3},
    },
    "early_terminate": {"type": "hyperband", "min_iter": 3, "eta": 2},
}

if not USE_WANDB:
    raise RuntimeError(
        "Sweeps require W&B. Add the WANDB_API_KEY secret (cell 2) and re-run.")

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)
print(f"Sweep created: {sweep_id}")
print(f"Watch it under project '{WANDB_PROJECT}' -> Sweeps on wandb.ai")

# Run N_TRIALS trials here. You can also paste the printed `wandb agent ...`
# command into a second Kaggle session to run trials in parallel against the same
# sweep — they all report to the same dashboard.
wandb.agent(sweep_id, function=train_run, count=N_TRIALS)

## 8. Reading the results (on the website)

Open project **`skin-lesion` -> Sweeps -> your sweep**. The three views that make
this worth more than a printout:

- **Parallel coordinates** — every trial is a line threading through the
  hyperparameter axes to its score. You *see* which regions are good.
- **Parameter importance** — ranks which hyperparameters actually moved the
  metric, and in which direction. This is your Optuna importance plot.
- **Best run** — sorted automatically. Copy its `ft_lr`/`head_lr`/`tau`/etc. into
  `kaggle_training.ipynb`'s config cell and retrain at full resolution.